<a href="https://www.kaggle.com/code/kohkaichunsamuel/deepseek-tir?scriptVersionId=296701775" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Setting Up VLLM Package to Run the LLM

We require to set up the v-LLM package that runs on Python 3.12.12.

Source from: https://www.kaggle.com/code/haradibots/installing-vllm-offline-fix

In [ ]:
import os
!mkdir /kaggle/working/vllm_312_wheels
# This downloads exactly what Python 3.12 needs
!pip download vllm==0.9.2 -d /kaggle/working/vllm_312_wheels
# Zip it so you can download it easily
!zip -r vllm_312.zip /kaggle/working/vllm_312_wheels

In [ ]:
from IPython.display import FileLink
FileLink(r'vllm_312.zip')

In [ ]:
import os
print(os.listdir('/kaggle/input'))

In [ ]:
!find /kaggle/input -name "*.whl"

In [ ]:
# Install directly from the folder shown in your screenshot
!pip install --no-index --find-links=/kaggle/input/vllm-4 vllm

In [ ]:
# !pip uninstall -y torch
# # !pip install --no-index --find-links=/kaggle/input/vllm-whl -U vllm

### Testing the VLLM

In [1]:
import vllm
print(f"vLLM version {vllm.__version__} successfully installed!")

vLLM version 0.9.2 successfully installed!


In [2]:
!python --version

Python 3.12.12


In [3]:
import transformers
print(transformers.__version__)

4.53.3


In [4]:
# 1. Force-install the compatible version from the internet
!pip install "transformers<4.54.0"

^C
ERROR: Operation cancelled by user


In [5]:
from vllm import LLM, SamplingParams
import pandas as pd
from tqdm import tqdm
import gc
import re
import sys
import subprocess
from collections import defaultdict, Counter
import numpy as np
from transformers import (AutoModelForCausalLM,
    AutoTokenizer,
    set_seed)
import torch
import math

llm = LLM(model="/kaggle/input/deepseek-math",
          dtype='half',
          enforce_eager=True,
          gpu_memory_utilization=0.99,
          swap_space=4,
          max_model_len=2048,
          kv_cache_dtype="fp8_e5m2",
          tensor_parallel_size=1)

llm

2026-02-07 15:02:41.536168: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770476561.557877     362 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770476561.564606     362 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770476561.581617     362 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770476561.581639     362 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770476561.581641     362 computation_placer.cc:177] computation placer alr

INFO 02-07 15:02:46 [__init__.py:244] Automatically detected platform cuda.
INFO 02-07 15:03:04 [config.py:841] This model supports multiple tasks: {'classify', 'reward', 'generate', 'embed'}. Defaulting to 'generate'.
WARNING 02-07 15:03:04 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 02-07 15:03:04 [config.py:1472] Using max model len 2048
WARNING 02-07 15:03:04 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 02-07 15:03:04 [config.py:1593] Using fp8 data type to store kv cache. It reduces the GPU memory footprint and boosts the performance. Meanwhile, it may cause accuracy drop without a proper scaling factor
WARNING 02-07 15:03:08 [cuda.py:102] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 02-07 15:03:08 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.2) with config: model='/kaggle/input/deepseek-math', specu

[W207 15:03:09.186813488 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W207 15:03:09.187896467 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 02-07 15:04:16 [default_loader.py:272] Loading weights took 65.63 seconds
INFO 02-07 15:04:16 [model_runner.py:1203] Model loading took 12.8726 GiB and 65.861306 seconds
INFO 02-07 15:04:19 [worker.py:294] Memory profiling takes 2.16 seconds
INFO 02-07 15:04:19 [worker.py:294] the current vLLM instance can use total_gpu_memory (14.56GiB) x gpu_memory_utilization (0.99) = 14.42GiB
INFO 02-07 15:04:19 [worker.py:294] model weights take 12.87GiB; non_torch_memory takes 0.03GiB; PyTorch activation peak memory takes 0.95GiB; the rest of the memory reserved for KV Cache is 0.57GiB.
INFO 02-07 15:04:20 [executor_base.py:113] # cuda blocks: 154, # CPU blocks: 1092
INFO 02-07 15:04:20 [executor_base.py:118] Maximum concurrency for 2048 tokens per request: 1.20x
INFO 02-07 15:04:24 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 7.90 seconds


### Instantiating the Tokeniser for the Process Reward Model (PRM)

In [6]:
tokenizer = llm.get_tokenizer()

good_token = '+'
bad_token = '-'
step_tag = 'ки'

prm_tokenizer = AutoTokenizer.from_pretrained('/kaggle/input/math-shepherd-mistral-7b-prm')

prm_tokenizer

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggin

LlamaTokenizerFast(name_or_path='/kaggle/input/math-shepherd-mistral-7b-prm', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

### Encoding the PRM

In [7]:
prm_candidate_tokens = prm_tokenizer.encode(f"{good_token} {bad_token}")[1:] # [648, 387]
step_tag_id = prm_tokenizer.encode(f"{step_tag}")[-1] # 12902

step_tag_id

12902

### Initialising the PRM

In [8]:
prm_model = AutoModelForCausalLM.from_pretrained('/kaggle/input/math-shepherd-mistral-7b-prm',
                                                 torch_dtype=torch.float16,
                                                 device_map="balanced_low_0").eval()
prm_model

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_e

In [9]:
import pandas as pd
from tqdm import tqdm
PRIVATE = True

df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv')
df.head()

,id,problem
0,000aaa,What is $1-1$?
1,111bbb,What is $0\times10$?
2,222ccc,Solve $4+x=4$ for $x$.


In [10]:
if len(df) < 5:
    df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-prize/train.csv')
    PRIVATE = False
df.head()

,id,problem,answer
0,229ee8,"Let $k, l > 0$ be parameters. The parabola $y ...",52
1,246d26,Each of the three-digits numbers $111$ to $999...,250
2,2fc4ad,Let the `sparkle' operation on positive intege...,702
3,430b63,What is the minimum value of $5x^2+5y^2-8xy$ w...,800
4,5277ed,There exists a unique increasing geometric seq...,211


### Building the Naive Parser

This function only takes out the final answer (number) from the long answer the model generates

In [11]:
def naive_parse(answer):
    out = []
    start = False
    end = False
    for l in reversed(list(answer)):
        if l in '0123456789' and not end:
            start = True
            out.append(l)
        else:
            if start:
                end = True
        
    out = reversed(out)
    return ''.join(out)

### Top N Strings

In [12]:
import re
import sys
import os
import subprocess

def top_n_strings(input_list, n):
    # Sort the list based on the prob values in descending order
    sorted_list = sorted(input_list, key=lambda x: x[1], reverse=True)

    # Get the top n elements
    top_n = sorted_list[:n]

    # Extract the strings from the top n elements
    result = [item[0] for item in top_n]

    return result

## Helper Functions for Phase 2 - Tool Integrated Reasoning (TIR)

### Process Output

Cuts the tiny code out and saves into a code.py block to run. If it fails, it gives an error.

In [13]:
def process_output(output):
    result = output
    
    try:
        code = output.split('```')[1][7:]

        with open('code.py', 'w') as fout:
            fout.write(code)

        batcmd = 'timeout 7 ' + sys.executable + ' code.py'
        try:
            shell_output = subprocess.check_output(batcmd, shell=True).decode('utf8')
            print(shell_output)
            code_output = round(float(eval(shell_output))) % 1000
        except:
            code_output = -1
            
        if os.path.exists('code.py'):
            os.remove('code.py')

        print('CODE RESULTS', code_output)
    
    except Exception as e:
        print(e)
        print('ERROR PARSING')
        code_output = -1
    
    try:
        result_output = re.findall(r'\\boxed\{(.*)\}', result)

        print('BOXED', result_output)
        if not len(result_output):
            result_output = naive_parse(result)
        else:
            result_output = result_output[-1]

        print('BOXED', result_output)
        if not len(result_output):
            result_output = -1
        
        else:
            result_output = round(float(eval(result_output))) % 1000
    
    except Exception as e:
        print(e)
        print('ERROR PARSING')
        result_output = -1
    
    return result_output, code_output

### Function for Process Reward Model (PRM)

- Takes all the different ideas the AI has, and gives each one a score (good or bad).
  

In [ ]:
batch_size = 1

def eval_prm(candidates):
    # Initialize a list to store all the log probabilities
    all_log_probs = []

    # Process the candidates in batches
    for i in range(0, len(candidates), batch_size):
        # Select a batch of candidates
        batch_candidates = candidates[i:i + batch_size]

        # Encode all candidates into a batch of input IDs
        encoded_inputs = [prm_tokenizer.encode(candidate, return_tensors="pt") for candidate in batch_candidates]

        # Pad the encoded inputs to the same length
        max_length = max([input_id.shape[1] for input_id in encoded_inputs])  # Find the longest sequence
        padded_inputs = [
            torch.nn.functional.pad(input_id, (0, max_length - input_id.size(1)), value=prm_tokenizer.pad_token_id) for
            input_id in encoded_inputs]
        input_ids = torch.cat(padded_inputs, dim=0).to("cuda:1")  # Concatenate the padded inputs into a tensor

        with torch.no_grad():
            logits = prm_model(input_ids).logits[:, :, prm_candidate_tokens]

            scores = logits.softmax(dim=-1)[:, :, 0].squeeze()

            # Extract log probabilities for the specific candidate tokens
            log_probs = scores.log()

            if batch_size == 1:
                batch_log_probs = log_probs[-1].flatten()
            else:
                batch_log_probs = log_probs[:, -1].flatten()

            # Collect the log probabilities from this batch
            all_log_probs.extend(batch_log_probs.cpu().tolist())

    return all_log_probs

### Getting Stop Words

In [ ]:
stop_words = [tokenizer.eos_token if tokenizer is not None and tokenizer.eos_token is not None else '</s>']
stop_words.append("\n")

tool_stop_words = [tokenizer.eos_token if tokenizer is not None and tokenizer.eos_token is not None else '</s>']
tool_stop_words.append("```output")

print("Stop words:", stop_words)
print("Tool stop words:", tool_stop_words)

In [ ]:
sampling_params = SamplingParams(temperature=1.0,
                                 max_tokens=256,
                                 stop=stop_words)

tool_sampling_params = SamplingParams(temperature=1.0,
                                      max_tokens=2048,
                                      stop=tool_stop_words)


### Prompts for DeepSeek to Solve

In [ ]:
cot_instruction = "\nPlease reason step by step, and put your final \
answer within \\boxed{}."

tool_instruction = '\nPlease integrate natural language reasoning \
with programs to solve the problem above, and put your final answer \
within \\boxed{}.'

### The Main Loop

1. It lets the AI solve the problem multiple times, attaining about 48 different solutions.
2. It uses the Judge to keep the best ones and eliminate the bad ones.
3. If the AI is stuck, it will switch to Python coding using the Python interpreter. (TIR is used here).
4. Finally, it counts all the answers it got and picks the one that appeared the most (majority voting).

### Main Loop

In [ ]:
n = 4
all_prompts = []
total_results = []
total_answers = []

for i in tqdm(range(len(df))):
    id_ = df['id'].loc[i]
    problem = df['problem'].loc[i]

    messages = [
        {
            "role": "user",
            "content": problem + cot_instruction
        }
    ]

    base_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )
    current_level = 0

    current_level_nodes = [(base_prompt, 0)]  # Tuple of (node, cumulative_logprob)
    completed_paths = []

    while len(completed_paths) < n:
        # Prepare batch for generation
        batch_prompts = [node for node, _ in current_level_nodes]
        batch_responses = llm.generate(batch_prompts*16, sampling_params)  # Generate for all nodes in a batch

        prm_inputs = []  # To collect all candidates for reward model evaluation
        mapping = []  # To map back reward model scores to corresponding nodes

        # Collect candidates for reward model evaluation
        for parent_node, parent_cum_logprob in current_level_nodes:
            for candidate in batch_responses:
                if parent_node + candidate.outputs[0].text + "\n" not in [prm_input[1] for prm_input in prm_inputs]:
                    new_node = parent_node + candidate.outputs[0].text + "\n"
                    cumulative_tokens = len(candidate.prompt_token_ids) + len(candidate.outputs[0].token_ids)
                    prm_inputs.append((new_node[:-1] + " " + step_tag, new_node, parent_cum_logprob, cumulative_tokens))
                    mapping.append(len(prm_inputs) - 1)  # Store the index for mapping back the score

        # Batch reward model evaluation
        prm_scores = eval_prm([prm_input for prm_input, _, _, _ in prm_inputs])
        next_level_nodes = []

        # Distribute candidates back to their parent nodes
        for idx, (_, node, parent_cum_logprob, cumulative_tokens) in enumerate(prm_inputs):
            score = prm_scores[idx]  # Get the corresponding score
            new_cum_logprob = parent_cum_logprob + score  # Update cumulative log probability

            # Check for completions and sufficient score
            if "```" in node or "\\boxed" in node.split("\n\n")[-1]:
                completed_paths.append((node, new_cum_logprob))
            elif score > math.log(0.8) and cumulative_tokens <= 2000:  # Threshold check
                next_level_nodes.append((node, new_cum_logprob))

        # Prune to keep only the top 'n' candidates based on cumulative log probability
        next_level_nodes.sort(key=lambda x: x[1], reverse=True)  # Sort nodes by their cumulative log probability
        current_level_nodes = next_level_nodes[:n]  # Keep only the top 'n' nodes
        current_level += 1

        # If we already have 'n' completed paths, no need to continue
        if len(completed_paths) >= n:
            break
        if not current_level_nodes or current_level > 20:
            if not completed_paths:
                messages = [
                    {
                        "role": "user",
                        "content": problem + tool_instruction
                    }
                ]

                base_prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False
                ) + "```python\n"

                raw_outputs = llm.generate([base_prompt]*8, tool_sampling_params)

                for response in raw_outputs:
                    completed_paths.append("```python\n" + response.outputs[0].text)
            break

    results = []
    answers = []

    for path in completed_paths:
        try:
            if "```python\n" in path:
                result_output, code_output = process_output(path)
                gc.collect()
                
                results.append(code_output)
                answers.append(code_output)
            else:
                raw_output = path[0].split("\n\n")[-1]
                result_output, code_output = process_output(raw_output)
                gc.collect()

                results.append(result_output)
                answers.append(result_output)

        except Exception as e:
            print(e)
            result_output, code_output = -1, -1
            logprob = -10000

            results.append(result_output)
            answers.append(result_output)

    total_results.append(results)
    total_answers.append(answers)

### Obtain Final Answers

In [ ]:
import numpy as np
from collections import Counter

final_answers = []

for a, b in zip(total_answers, total_results):
    a = np.array(a)
    b = np.array(b)
    a[a < 0] = b[a < 0]

    pred = Counter(a.tolist()).most_common(2)

    try:
        ans = pred[0][0] if not pred[0][0] < 0 else pred[1][0]
    except:
        if len(a) == 1:
            ans = a[0]
        else:
            ans = 0

    final_answers.append(ans)
    print(ans)

In [ ]:
df['answer'] = final_answers

In [ ]:
df

In [ ]:
df[['id','answer']].to_csv("submission.csv", header=True, index=False)

In [ ]:
df[['id','answer']].head()

### Loading DeepSeek Prover using Monte Carlo Tree Search

In [22]:
"""
Monte Carlo Tree Search (MCTS) implementation for mathematical reasoning
"""
from typing import List, Dict, Any, Optional, Tuple
import numpy as np
from dataclasses import dataclass
import json
from pathlib import Path

@dataclass
class MCTSConfig:
    exploration_weight: float = 1.0
    max_simulations: int = 1000
    max_depth: int = 10

class MCTSNode:
    def __init__(self, state: str, parent: Optional['MCTSNode'] = None, action: Optional[str] = None):
        self.state = state
        self.parent = parent
        self.action = action
        self.children: List[MCTSNode] = []
        self.visits = 0
        self.value = 0.0
        self.untried_actions: List[str] = []
        
    def add_child(self, action: str, state: str) -> 'MCTSNode':
        """Add a child node with the given action and state."""
        child = MCTSNode(state=state, parent=self, action=action)
        self.children.append(child)
        if action in self.untried_actions:
            self.untried_actions.remove(action)
        return child

    def update(self, reward: float) -> None:
        """Update node statistics with new reward."""
        self.visits += 1
        self.value += (reward - self.value) / self.visits
        
    def get_ucb_score(self, exploration_weight: float) -> float:
        """Calculate UCB1 score for this node."""
        if self.visits == 0:
            return float('inf')
        exploitation = self.value / self.visits
        exploration = exploration_weight * np.sqrt(2 * np.log(self.parent.visits) / self.visits)
        return exploitation + exploration

    def is_terminal(self) -> bool:
        """Check if this node represents a terminal state."""
        # Implementation depends on problem domain
        return False

    def get_possible_actions(self) -> List[str]:
        """Get list of possible actions from this state."""
        # Implementation depends on problem domain
        return []

class MCTS:
    def __init__(self, config: Optional[MCTSConfig] = None):
        self.config = config or MCTSConfig()
        
    @classmethod
    def from_config_file(cls, config_path: str) -> 'MCTS':
        """Create MCTS instance from config file."""
        with open(config_path, 'r') as f:
            config_data = json.load(f)
        config = MCTSConfig(**config_data['mcts'])
        return cls(config)
        
    def select_action(self, node: MCTSNode) -> Tuple[MCTSNode, str]:
        """Select the best child node using UCB1."""
        if not node.children:
            return node, ""
            
        ucb_scores = [
            child.get_ucb_score(self.config.exploration_weight)
            for child in node.children
        ]
        selected_child = node.children[np.argmax(ucb_scores)]
        return selected_child, selected_child.action
    
    def expand(self, node: MCTSNode) -> Tuple[MCTSNode, str]:
        """Expand the current node with a new child."""
        if not node.untried_actions:
            node.untried_actions = node.get_possible_actions()
            
        if not node.untried_actions:
            return node, ""
            
        action = np.random.choice(node.untried_actions)
        new_state = self.apply_action(node.state, action)
        child = node.add_child(action, new_state)
        return child, action
    
    def simulate(self, state: str, depth: int = 0) -> float:
        """Run a simulation from the current state."""
        if depth >= self.config.max_depth or self.is_terminal_state(state):
            return self.evaluate_state(state)
            
        actions = self.get_possible_actions(state)
        if not actions:
            return self.evaluate_state(state)
            
        action = np.random.choice(actions)
        new_state = self.apply_action(state, action)
        return self.simulate(new_state, depth + 1)
    
    def backpropagate(self, node: MCTSNode, reward: float) -> None:
        """Update the values up the tree."""
        while node is not None:
            node.update(reward)
            node = node.parent
            
    def search(self, root_state: str) -> Tuple[str, List[Dict[str, Any]]]:
        """Perform MCTS search to find the best action sequence."""
        root = MCTSNode(state=root_state)
        trajectory = []
        
        for _ in range(self.config.max_simulations):
            node = root
            
            # Selection
            while node.children and not node.untried_actions:
                node, action = self.select_action(node)
                trajectory.append({
                    "state": node.state,
                    "action": action,
                    "value": node.value,
                    "visits": node.visits
                })
                
            # Expansion
            if not node.is_terminal():
                node, action = self.expand(node)
                trajectory.append({
                    "state": node.state,
                    "action": action,
                    "value": node.value,
                    "visits": node.visits
                })
                
            # Simulation
            reward = self.simulate(node.state)
            
            # Backpropagation
            self.backpropagate(node, reward)
            
        # Return best action sequence
        best_child = max(root.children, key=lambda c: c.visits)
        return best_child.action, trajectory
    
    def apply_action(self, state: str, action: str) -> str:
        """Apply an action to a state to get the next state."""
        # Implementation depends on problem domain
        pass
    
    def evaluate_state(self, state: str) -> float:
        """Evaluate the value of a terminal state."""
        # Implementation depends on problem domain
        pass
    
    def is_terminal_state(self, state: str) -> bool:
        """Check if a state is terminal."""
        # Implementation depends on problem domain
        pass
    
    def get_possible_actions(self, state: str) -> List[str]:
        """Get possible actions for a state."""
        # Implementation depends on problem domain
        pass


### Process Preference Model (PPM)

In [23]:
"""
Process Preference Model (PPM) for evaluating reasoning steps
"""
from typing import List, Dict, Any, Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import json
from pathlib import Path

@dataclass
class PPMConfig:
    input_dim: int = 768
    hidden_dim: int = 256
    learning_rate: float = 0.001
    batch_size: int = 32

class StepEncoder(nn.Module):
    """Encodes reasoning steps into fixed-size vectors."""
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)

class ProcessPreferenceModel(nn.Module):
    def __init__(self, config: Optional[PPMConfig] = None):
        super().__init__()
        self.config = config or PPMConfig()
        
        # Step encoder
        self.step_encoder = StepEncoder(
            self.config.input_dim,
            self.config.hidden_dim
        )
        
        # Value head
        self.value_head = nn.Sequential(
            nn.Linear(self.config.hidden_dim, self.config.hidden_dim // 2),
            nn.LayerNorm(self.config.hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(self.config.hidden_dim // 2, 1)
        )
        
        # Initialize optimizer
        self.optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.config.learning_rate
        )
        
    @classmethod
    def from_config_file(cls, config_path: str) -> 'ProcessPreferenceModel':
        """Create PPM instance from config file."""
        with open(config_path, 'r') as f:
            config_data = json.load(f)
        config = PPMConfig(**config_data['ppm'])
        return cls(config)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the model."""
        encoded = self.step_encoder(x)
        value = self.value_head(encoded)
        return value
    
    def evaluate_step(self, step: str, embedder: Any) -> float:
        """Evaluate a single reasoning step."""
        self.eval()
        with torch.no_grad():
            # Convert step to embedding using provided embedder
            embedding = embedder.encode(step)
            embedding_tensor = torch.FloatTensor(embedding).unsqueeze(0)
            
            # Get value prediction
            value = self(embedding_tensor)
            return value.item()
    
    def train_step(self, 
                  preferred_steps: List[str],
                  non_preferred_steps: List[str],
                  embedder: Any) -> float:
        """Train the model on a batch of preferred vs non-preferred steps."""
        self.train()
        
        # Convert steps to embeddings
        preferred_embeddings = torch.FloatTensor([
            embedder.encode(step) for step in preferred_steps
        ])
        non_preferred_embeddings = torch.FloatTensor([
            embedder.encode(step) for step in non_preferred_steps
        ])
        
        # Get value predictions
        preferred_values = self(preferred_embeddings)
        non_preferred_values = self(non_preferred_embeddings)
        
        # Compute preference loss (preferred steps should have higher values)
        loss = F.relu(non_preferred_values - preferred_values + 1.0).mean()
        
        # Backpropagation
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return loss.item()
    
    def save_model(self, path: str) -> None:
        """Save model state to file."""
        torch.save({
            'model_state_dict': self.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'config': self.config
        }, path)
    
    def load_model(self, path: str) -> None:
        """Load model state from file."""
        checkpoint = torch.load(path)
        self.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.config = checkpoint['config']
        
class PPMTrainer:
    def __init__(self, model: ProcessPreferenceModel):
        self.model = model
        
    def train(self, 
             training_data: List[Dict[str, Any]], 
             embedder: Any,
             num_epochs: int = 100,
             validation_data: Optional[List[Dict[str, Any]]] = None) -> Dict[str, List[float]]:
        """Train the PPM model on a dataset of preferred reasoning trajectories."""
        train_losses = []
        val_losses = []
        
        for epoch in range(num_epochs):
            # Training
            epoch_loss = 0.0
            for batch in self._create_batches(training_data, self.model.config.batch_size):
                loss = self.model.train_step(
                    batch['preferred'],
                    batch['non_preferred'],
                    embedder
                )
                epoch_loss += loss
            train_losses.append(epoch_loss / len(training_data))
            
            # Validation
            if validation_data:
                val_loss = self._validate(validation_data, embedder)
                val_losses.append(val_loss)
                
        return {
            'train_losses': train_losses,
            'val_losses': val_losses if validation_data else []
        }
    
    def _create_batches(self, 
                       data: List[Dict[str, Any]], 
                       batch_size: int) -> List[Dict[str, List[str]]]:
        """Create batches from training data."""
        batches = []
        for i in range(0, len(data), batch_size):
            batch_data = data[i:i + batch_size]
            batch = {
                'preferred': [d['preferred'] for d in batch_data],
                'non_preferred': [d['non_preferred'] for d in batch_data]
            }
            batches.append(batch)
        return batches
    
    def _validate(self, 
                 validation_data: List[Dict[str, Any]], 
                 embedder: Any) -> float:
        """Compute validation loss."""
        self.model.eval()
        total_loss = 0.0
        
        with torch.no_grad():
            for batch in self._create_batches(validation_data, self.model.config.batch_size):
                preferred_embeddings = torch.FloatTensor([
                    embedder.encode(step) for step in batch['preferred']
                ])
                non_preferred_embeddings = torch.FloatTensor([
                    embedder.encode(step) for step in batch['non_preferred']
                ])
                
                preferred_values = self.model(preferred_embeddings)
                non_preferred_values = self.model(non_preferred_embeddings)
                
                loss = F.relu(non_preferred_values - preferred_values + 1.0).mean()
                total_loss += loss.item()
                
        return total_loss / len(validation_data)


In [36]:
config_files = {"mcts": {
        "exploration_weight": 1.0,
        "max_simulations": 1000,
        "max_depth": 10
    },
    "ppm": {
        "input_dim": 768,
        "hidden_dim": 256,
        "learning_rate": 0.001,
        "batch_size": 32
    },
}

In [46]:
from types import SimpleNamespace

ppm_config = SimpleNamespace(**config_files['ppm'])
ppm = ProcessPreferenceModel(ppm_config)

mcts_config = SimpleNamespace(**config_files['mcts'])
mcts = MCTS(mcts_config)

In [51]:
import os
import plotly.express as px
import pandas as pd

def solve_math(problem: str):
    action, trajectory = mcts.search(problem)
    steps, scores = [], []

    for step in trajectory:
        score = ppm.evaluate_step(step['state'], llm)
        steps.append(step['state'])
        scores.append(score)
    return steps, scores
    

target_problem = "If $x^2 + y^2 = 25$ and $xy = 12$, find $(x+y)^2$."

print(f"Solving: {target_problem}\n" + "="*50)
solution_steps, confidence_scores = solve_math(target_problem)

Solving: If $x^2 + y^2 = 25$ and $xy = 12$, find $(x+y)^2$.


TypeError: unsupported operand type(s) for -: 'NoneType' and 'float'

In [50]:
llm

### Another Attempt

https://pub.aimind.so/using-llm-rl-with-mcts-for-solving-mathematical-problems-9c0f2935f1b6

In [55]:
class Node:
    def __init__(self, equation_state, parent=None):
        self.equation_state = equation_state
        self.parent = parent
        self.children = []
        self.visits = 0
        self.value = 0.0

In [72]:
# First, define your sampling parameters (do this once outside the function)
sampling_params = SamplingParams(
    temperature=0.7, 
    max_tokens=2048, 
    stop=["\n"]  # This forces the LLM to provide only ONE step
)

def generate_next_steps(equation_state):
    general_prompt = "You are a helpful assistant skilled in solving mathematical problems."
    # 1. Format the prompt correctly for DeepSeek
    prompt = f"System: {general_prompt}\nUser: Given the function '{equation_state}', provide the next best step to find its derivative. Provide only the step without asking questions.\nAssistant:"
    
    # 2. Use the .generate() method of the vLLM object
    # vLLM expects a list, so we wrap prompt in []
    outputs = llm.generate([prompt], sampling_params, use_tqdm=False)
    
    # 3. Extract the text from the first output
    generated_text = outputs[0].outputs[0].text
    
    return generated_text.strip()

In [78]:
# Use a separate SamplingParams for evaluation to keep it "focused"
eval_params = SamplingParams(
    temperature=0,      # We want deterministic scoring, no "creativity" here
    max_tokens=50,      # Scores should be short
    stop=["\n"]
)

def evaluate_step(equation_state, step):
    general_prompt = "You are a helpful assistant skilled in solving mathematical problems."
    # 1. Construct a strict prompt for scoring
    prompt = (
        f"System: {general_prompt}\n"
        f"User: Evaluate the following step in the derivative of '{equation_state}':\n"
        f"Step: '{step}'\n"
        f"Provide a numerical likelihood (0 to 100%) that this step is mathematically correct. "
        f"Respond ONLY with the percentage number.\n"
        f"Assistant:"
    )
    
    # 2. Generate with vLLM
    outputs = llm.generate([prompt], eval_params, use_tqdm=False)
    raw_output = outputs[0].outputs[0].text.strip()
    
    # 3. Clean the output to get a float (e.g., "85%" -> 0.85)
    try:
        # Extract digits using regex in case the LLM adds text
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", raw_output)
        if numbers:
            # Convert to a 0.0 - 1.0 scale for MCTS
            score = float(numbers[0]) / 100.0
            return max(0.0, min(1.0, score)) # Clamp between 0 and 1
        return 0.0
    except:
        return 0.0

In [79]:
def mcts_search(root, iterations):
    for _ in range(iterations):
        node = root
        
        # 1. Selection: Traverse until reaching a leaf node
        while node.children:
            node = select_best_node(node)
        
        # 2. Expansion: Generate possible next steps
        # Check if the node is terminal or already solved before expanding
        possible_steps = generate_next_steps(node.equation_state).split("\n")
        
        if possible_steps:
            for step in possible_steps:
                new_equation_state = step.strip()
                if new_equation_state:  # Avoid empty strings
                    child_node = Node(new_equation_state, parent=node)
                    node.children.append(child_node)
            
            chosen_child = random.choice(node.children)
            evaluation = evaluate_step(chosen_child.equation_state, chosen_child.equation_state)
            evaluation_score = parse_evaluation_score(evaluation)
            
            backpropagate(chosen_child, evaluation_score)
    
    return select_best_node(root)

In [81]:
import re

def parse_evaluation_score(evaluation):
    """
    Extracts a numeric score from the LLM's text output and 
    normalizes it to a 0.0 - 1.0 range.
    """
    if evaluation is None:
        return 0.0
    
    try:
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", str(evaluation))
        
        if not numbers:
            return 0.0 
        
        score = float(numbers[0])
        
        if score > 1.0:
            score = score / 100.0
            
        return max(0.0, min(1.0, score))
        
    except (ValueError, TypeError):
        return 0.0

In [83]:
def print_tree(node, depth=0):
    """
    Recursively prints the MCTS tree structure.
    """
    indent = "  " * depth
    stats = f"[V: {node.value:.2f}/N: {node.visits}]"
    print(f"{indent}└── {node.equation_state} {stats}")
    
    for child in node.children:
        print_tree(child, depth + 1)

In [84]:
import random
def backpropagate(node, score):
    while node is not None:
        node.visits += 1
        node.value += score
        node = node.parent

# --- INITIALIZATION AND EXECUTION ---

initial_equation = "f(x) = 3x^4 - 5x^3 + 2x - 7"
root_node = Node(initial_equation)

# Perform MCTS search with the logic defined in your previous cells
# Note: Ensure mcts_search is defined before running this
best_solution_node = mcts_search(root_node, iterations=10)

# Print all nodes in the tree
print("All nodes explored by MCTS:")
print_tree(root_node)

# Print the best solution found
print("\nBest solution path found by MCTS:")
solution_path = []
node = best_solution_node

# Trace back from the best leaf to the root
while node:
    solution_path.append(node.equation_state)
    node = node.parent

# Reverse the path so it reads from Start -> Finish
solution_path.reverse()
print(" -> ".join(solution_path))

All nodes explored by MCTS:
└── f(x) = 3x^4 - 5x^3 + 2x - 7 [V: 10.00/N: 10]
  └── To find the derivative of the function f(x) = 3x^4 - 5x^3 + 2x - 7, we will apply the power rule for each term. The power rule states that if f(x) = x^n, then f'(x) = n*x^(n-1). [V: 10.00/N: 10]
    └── To find the derivative of the function f(x) = 3x^4 - 5x^3 + 2x - 7, we will apply the power rule for each term. [V: 9.00/N: 9]
      └── To find the derivative of the function f(x) = 3x^4 - 5x^3 + 2x - 7, we will apply the power rule for each term. The power rule states that if f(x) = x^n, then f'(x) = n * x^(n-1). [V: 8.00/N: 8]
        └── To find the derivative of the function f(x) = 3x^4 - 5x^3 + 2x - 7, we will apply the power rule for each term. The power rule states that if f(x) = x^n, then f'(x) = n * x^(n-1). [V: 7.00/N: 7]
          └── To find the derivative of the function f(x) = 3x^4 - 5x^3 + 2x - 7, we will apply the power rule for each term. [V: 6.00/N: 6]
            └── To find the deriva